# Tool-calling SFT

`tools=` is not an API feature. It's a prompt, a fine-tuned habit, and a parser.

*Nobody taught the model to "call functions". They taught it that after a `<tools>` block, the
next thing you write is `<tool_call>{…}</tool_call>` — and wrote a parser to catch it.*

> **Draft — §3 onward is unrun.** §1–2 are verified (tokenizer only). Training needs torch;
> written against `trl` 1.8 / `transformers` 5.14. Last cell lists what to check on first run.

## 0. Setup

`uv sync` pulls torch — heavy. §1–2 run anywhere; §4 wants a GPU.

In [ ]:
import json
from pathlib import Path

from datasets import Dataset
from transformers import AutoTokenizer
from trl import SFTConfig, SFTTrainer

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small enough for CPU/MPS; its template is tool-aware
tok = AutoTokenizer.from_pretrained(MODEL)

TOOLS = [
    {"type": "function", "function": {
        "name": "multiply",
        "description": "Multiply two integers exactly.",
        "parameters": {"type": "object", "required": ["a", "b"], "properties": {
            "a": {"type": "integer"}, "b": {"type": "integer"}}}}},
    {"type": "function", "function": {
        "name": "current_time",
        "description": "Current wall-clock time in an IANA timezone.",
        "parameters": {"type": "object", "required": ["timezone"], "properties": {
            "timezone": {"type": "string", "description": "IANA name, e.g. 'Asia/Bangkok'"}}}}},
]

## 1. `tools=` is a prompt

The chat template pastes your schema into a system message. That's the whole mechanism.

In [ ]:
USER = [{"role": "user", "content": "What is 47281 * 918?"}]

bare = tok.apply_chat_template(USER, add_generation_prompt=True, tokenize=False)
shown = tok.apply_chat_template(USER, tools=TOOLS, add_generation_prompt=True, tokenize=False)

print(shown)
print("tokens:", len(tok(bare)["input_ids"]), "->", len(tok(shown)["input_ids"]))

## 2. The target — what a row teaches

The assistant turn you *want*. Rendering it shows the exact string SFT will score.

In [ ]:
CALL = {"role": "assistant", "content": "", "tool_calls": [
    {"type": "function", "function": {"name": "multiply",
                                      "arguments": {"a": 47281, "b": 918}}}]}

full = tok.apply_chat_template(USER + [CALL], tools=TOOLS, tokenize=False)
print(repr(full[len(shown):]))

## 3. The dataset

`data.jsonl` holds the examples — one `{question, name, arguments}` per line. Edit that file to
add training data; this cell only reshapes it into TRL's prompt/completion format.

In [ ]:
def to_row(ex):
    """{question, name, arguments} -> TRL prompt/completion row."""
    return {
        "prompt": [{"role": "user", "content": ex["question"]}],
        "completion": [{"role": "assistant", "content": "", "tool_calls": [
            {"type": "function", "function": {"name": ex["name"],
                                              "arguments": ex["arguments"]}}]}],
        "tools": TOOLS,
    }


raw = [json.loads(l) for l in Path("data.jsonl").read_text().splitlines() if l.strip()]
train = Dataset.from_list([to_row(ex) for ex in raw])

print(train)
print(f"{len(raw)} rows |", {ex["name"] for ex in raw})
print(json.dumps(train[0]["completion"], indent=1))

## 4. Train

`completion_only_loss=True` is the point: score the call, not the prompt. TRL does the `-100`
bookkeeping you'd otherwise hand-roll.

In [ ]:
cfg = SFTConfig(
    output_dir="out",
    max_steps=20,
    per_device_train_batch_size=1,
    learning_rate=2e-5,
    completion_only_loss=True,   # loss on the target only
    max_length=512,
    logging_steps=5,
    save_strategy="no",
    report_to=[],
)

trainer = SFTTrainer(model=MODEL, args=cfg, train_dataset=train)
trainer.train()

## 5. Did it learn the syntax?

Not "is it right" — 10 rows and 20 steps learns nothing useful. Only: does it now emit the
`<tool_call>` shape the parser expects?

In [ ]:
import torch

prompt = tok.apply_chat_template(
    [{"role": "user", "content": "What is 8 times 9?"}],
    tools=TOOLS, add_generation_prompt=True, tokenize=False,
)
ids = tok(prompt, return_tensors="pt").to(trainer.model.device)
with torch.no_grad():
    out = trainer.model.generate(**ids, max_new_tokens=64, do_sample=False)

print(tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=False))

## Weaknesses

- **It's imitation, never a constraint.** SFT raises the probability of the format; it can't
  make it certain. Flaky tool choice at inference is a habit, not a broken schema.
- **The format is per-model.** Qwen emits `<tool_call>{…}</tool_call>`; Llama 3.1 emits
  `<function=name>{…}</function>`. Train on one template and the weights encode *that* syntax —
  the server's parser must match, and a mismatch is the `content` leak seen in
  `agt_tool_calling`.
- **Rendering is silently optional.** `NousResearch/Meta-Llama-3.1-8B-Instruct`'s template
  ignores `tools=` and drops your schema on the floor — no error, just a prompt with no tools
  in it. Always render and read it before you trust it.
- **Schemas are not free.** One tool cost `43 -> 185` tokens. Every call pays for every tool you
  expose, used or not.
- **10 rows teaches syntax, not judgement.** Real tool-use SFT needs thousands of diverse
  traces — and negatives, where the right move is *not* to call. `data.jsonl` has no example of
  the model declining, so it can only learn "always call something".
- **Constrained decoding is the real fix.** Grammar / guided-JSON masks logits at inference so
  invalid output is unrepresentable — the only layer that makes the schema a contract.

## First-run checklist — what I could not verify

Verified: `trl` 1.8.0 / `torch` 2.13.0 / `transformers` 5.14.1 install on macOS (MPS available),
`SFTConfig` accepts `completion_only_loss` and `assistant_only_loss`, and §1–2's numbers
(`43 -> 185` tokens, 32-token target).

Unverified — check in this order:

1. **The `tools` column.** §3 passes `tools` beside `prompt`/`completion`, assuming TRL forwards
   it to `apply_chat_template`. If the rendered prompt lacks the `<tools>` block, TRL is
   dropping it — pre-render §1's `shown`/`full` into a `text` column instead.
2. **`SFTTrainer(model="…")` as a string.** Should load it; if not, pass a loaded
   `AutoModelForCausalLM`.
3. **Device.** Trainer should pick MPS/CUDA; add `use_cpu=True` to `SFTConfig` if MPS misbehaves.
4. **The mask.** Inspect `trainer.train_dataset[0]` for `completion_mask` and check its sum
   matches §2's 32-token target. That is the one number that proves the loss is on the call.